# Lesson 16: Fine-tuning language models

*Worked solutions for [the lesson page](../../lessons/16-fine-tuning.html). The exercise version without solutions is `../16-fine-tuning.ipynb`.*

Learn when fine-tuning is the right tool and when it is not, understand LoRA and QLoRA down to the matrix shapes and memory budget, fine-tune a small open model on your own task, then go further with DPO, distillation and adapter serving.

**Runs on:** A GPU runtime (a free Colab T4 is enough for the 0.5B model). The LoRA-from-scratch cell runs on CPU.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q torch transformers datasets peft trl accelerate bitsandbytes

## When to fine-tune, and when not to

> **Before you fine-tune, have an eval set**
>
> Without the evaluation discipline of lesson 15 you cannot tell whether fine-tuning helped, hurt, or just changed things. Measure the prompted baseline first. Fine-tuning that does not beat a well-prompted baseline on your eval set is wasted money.

## LoRA: low-rank adaptation

### LoRA from scratch

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
import torch, torch.nn as nn

class LoRALinear(nn.Module):
    """Wrap a frozen nn.Linear with a trainable low-rank update."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                                       # freeze W and b
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)    # (r, d_in)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))          # (d_out, r): zero, so no change at start
        self.scale = alpha / r

    def forward(self, x):                                                 # x: (..., d_in)
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale     # (..., d_out)

    @torch.no_grad()
    def merge(self) -> nn.Linear:
        """Fold the update into W for zero-overhead inference."""
        self.base.weight += (self.B @ self.A) * self.scale
        return self.base

layer = LoRALinear(nn.Linear(4096, 4096), r=16)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in layer.parameters())
print(trainable, total, f"{100 * trainable / total:.2f}%")   # 131072 16912384 0.78%

## Hands-on: fine-tune a small model

### The data

Training data for chat models is a list of conversations in the same chat format the model was instruction-tuned with:

*train.jsonl (one example per line, shown pretty-printed)*

```json
{"messages": [
  {"role": "system", "content": "Convert the technician note to JSON with keys instrument_id, action, parts, duration_min, follow_up."},
  {"role": "user", "content": "P-117 coastal. swapped desiccant + new belt on filter wheel, 45 min. check again in 2 wks"},
  {"role": "assistant", "content": "{\"instrument_id\": \"P-117\", \"action\": \"maintenance\", \"parts\": [\"desiccant\", \"filter wheel belt\"], \"duration_min\": 45, \"follow_up\": \"recheck in 2 weeks\"}"}
]}
```

### Training with Hugging Face TRL and PEFT

Run this on a GPU (a free Colab T4 is enough for a 0.5B model). Library APIs evolve quickly; if an argument name has changed, check the TRL documentation for your installed version.

Run in a terminal:

```bash
pip install -U transformers datasets peft trl accelerate bitsandbytes
```

In [ ]:
# Build a synthetic training set for the notes-to-JSON task, so the notebook runs without any API.
# In the hands-on exercise you replace this with distilled or hand-labelled data.
import json, random

random.seed(0)
SYSTEM = ("Convert the technician note to JSON with keys "
          "instrument_id, action, parts, duration_min, follow_up.")
JOBS = [
    ("maintenance", ["desiccant"], ["swapped desiccant", "new desiccant cartridge", "desiccant replaced"]),
    ("maintenance", ["filter wheel belt"], ["new belt on filter wheel", "replaced FW belt", "fw belt swapped"]),
    ("maintenance", ["desiccant", "filter wheel belt"], ["swapped desiccant + new belt on filter wheel",
                                                         "desiccant and fw belt both replaced"]),
    ("maintenance", ["entrance window"], ["cleaned entrance window", "window cleaned, lots of dust"]),
    ("repair", ["gps antenna"], ["replaced gps antenna", "new GPS antenna fitted"]),
    ("repair", ["cooler fan"], ["cooler fan dead, replaced", "swapped cooler fan"]),
    ("repair", ["modem"], ["swapped modem", "modem replaced (no signal)"]),
    ("firmware_update", [], ["firmware 2.3->2.4", "fw update to 2.4", "updated firmware to v2.4"]),
    ("inspection", [], ["visual check, nothing found", "routine inspection", "looked over, all fine"]),
]
FOLLOW = [(None, ["all ok", "", "done"]),
          ("recheck in 2 weeks", ["check again in 2 wks", "recheck in two weeks"]),
          ("return with spare belt", ["need to come back w/ spare belt", "return with spare belt"]),
          ("monitor data upload", ["keep an eye on uploads", "monitor upload tonight"])]
SITES = ["coastal", "urban", "desert", "alpine", ""]

def make_example():
    action, parts, phrasings = random.choice(JOBS)
    follow, follow_phrasings = random.choice(FOLLOW)
    minutes = random.choice([10, 15, 20, 30, 45, 60, 90, None])
    if minutes is None:
        dur = ""
    elif minutes >= 60 and random.random() < 0.5:
        dur = f"{minutes / 60:g}h"
    else:
        dur = random.choice([f"{minutes} min", f"{minutes}min", f"{minutes} mins"])
    iid = f"P-{random.randint(100, 299)}"
    pieces = [random.choice(phrasings), dur, random.choice(follow_phrasings)]
    pieces = [p for p in pieces if p]
    head = f"{iid} {random.choice(SITES)}".strip()
    note = f"{head}{random.choice(['. ', ': ', ' - '])}" + ", ".join(pieces)
    if random.random() < 0.3:
        note = note.lower()
    target = {"instrument_id": iid, "action": action, "parts": parts,
              "duration_min": minutes, "follow_up": follow}
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": note},
                         {"role": "assistant", "content": json.dumps(target)}]}

for name, n in [("train.jsonl", 400), ("val.jsonl", 60)]:
    with open(name, "w") as f:
        for _ in range(n):
            f.write(json.dumps(make_example()) + "\n")
print(open("train.jsonl").readline()[:300])

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"         # small instruction-tuned open model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

data = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear",                 # LoRA on every linear layer
    task_type="CAUSAL_LM",
)
args = SFTConfig(
    output_dir="notes-to-json-lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,               # effective batch of 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                                   # use bf16=True instead on Ampere or newer GPUs
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    peft_config=lora,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model("notes-to-json-lora")         # saves only the adapter: a few MB

For a larger model on limited memory, load it in 4-bit (QLoRA) instead:

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct",
                                             quantization_config=bnb, device_map="auto")
# then the same LoraConfig / SFTConfig / SFTTrainer, typically with gradient_checkpointing=True in SFTConfig

### Use and evaluate the result

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
tuned = PeftModel.from_pretrained(base, "notes-to-json-lora")

def convert(note: str, m) -> str:
    msgs = [{"role": "system", "content": "Convert the technician note to JSON with keys "
                                          "instrument_id, action, parts, duration_min, follow_up."},
            {"role": "user", "content": note}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(m.device)
    out = m.generate(ids, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

print(convert("P-204: firmware 2.3->2.4, rebooted, all ok. 15min", tuned))
merged = tuned.merge_and_unload()                # fold LoRA into the weights for deployment
merged.save_pretrained("notes-to-json-merged")

## Beyond supervised fine-tuning

### Preference tuning with DPO

In words: raise the probability of the chosen answer $y_w$ relative to the rejected one $y_l$, measured against a frozen reference model so the tuned model does not drift too far; $\beta$ controls how far. It is usually run after SFT.

*Shown for reference, not run here:*

```python
from trl import DPOConfig, DPOTrainer

prefs = load_dataset("json", data_files="prefs.jsonl")["train"]   # {"prompt", "chosen", "rejected"}
trainer = DPOTrainer(
    model=sft_model,                       # start from your SFT result
    args=DPOConfig(output_dir="dpo-out", beta=0.1, learning_rate=5e-6, num_train_epochs=1),
    train_dataset=prefs,
    processing_class=tokenizer,
    peft_config=lora,                      # with LoRA, the frozen base acts as the reference model
)
trainer.train()
```

### Hands-on: distil a large model into a small one

1. Write 30 realistic technician notes by hand (vary style, typos, abbreviations, and include some notes with missing information).
2. Use a large model via API to generate 400 more diverse notes and their JSON conversions. Review 40 at random and fix or drop bad ones.
3. Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` with LoRA as above.
4. On your 30 hand-written notes (the eval set, never trained on), measure: JSON validity, exact match per field, and latency for the base small model with a few-shot prompt, the fine-tuned small model, and the large model.
5. Write one paragraph recommending which to deploy, and why.

#### Solution: what to expect and how to grade it

In [ ]:
import json

FIELDS = ["instrument_id", "action", "parts", "duration_min", "follow_up"]

def grade(pred: str, gold: dict) -> dict:
    try:
        p = json.loads(pred)
    except json.JSONDecodeError:
        return {"valid": 0, **{f: 0 for f in FIELDS}}
    return {"valid": 1, **{f: int(p.get(f) == gold.get(f)) for f in FIELDS}}

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
val = [json.loads(l) for l in open("val.jsonl")]

def evaluate(m, n=60):
    totals = {"valid": 0, **{f: 0 for f in FIELDS}}
    for ex in val[:n]:
        note, gold = ex["messages"][1]["content"], json.loads(ex["messages"][2]["content"])
        for k, v in grade(convert(note, m), gold).items():
            totals[k] += v
    return {k: round(v / min(n, len(val)), 3) for k, v in totals.items()}

fresh_base = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
print("base model :", evaluate(fresh_base))
print("fine-tuned :", evaluate(merged))

Typical outcome: the base 0.5B model with a few-shot prompt produces valid JSON most of the time but gets fields wrong or invents values; the fine-tuned model is close to 100% valid and approaches the large model on `instrument_id`, `duration_min` and `parts`, while still trailing on free-text fields like `follow_up`. It runs on a laptop CPU at a fraction of the cost. A sound recommendation might be: deploy the fine-tuned small model, route notes it fails to parse (or flags as low confidence) to the large model, and keep adding corrected failures to the training set for the next round. The exact numbers matter less than having measured them on data the model never saw.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/16-fine-tuning.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.